# Lesson 5 - Creating an A2A Sequential Chain Agent with ADK

Now that you have two active agents (Policy Agent and Research Agent), you will orchestrate them. In this lesson, you will use Google ADK's `SequentialAgent` to create a workflow where a user's query is processed by the Research Agent first, and then the Policy Agent, or vice versa, based on the sequential definition. You will use `RemoteA2aAgent` to connect to the servers you started in previous lessons.

```mermaid
graph LR
    classDef orchestrator fill:#f9f,stroke:#333,stroke-width:2px;
    classDef agent fill:#e1f5fe,stroke:#0277bd,stroke-width:2px;
    classDef tool fill:#fff3e0,stroke:#ef6c00,stroke-width:1px,stroke-dasharray: 5 5;

    subgraph Sequential [Sequential Agent]
        direction LR
        
        HealthcareAgent["<b>Healthcare Research<br/>Agent</b>"]
        PolicyAgent["<b>Policy<br/>Agent</b>"]
        
        HealthcareAgent --> PolicyAgent
    end

    class HealthcareAgent,PolicyAgent agent;
```

## 5.1. Start the A2A Servers

Ensure both the Policy Agent (Terminal 1) and Research Agent (Terminal 2) are running.
- Open two terminals as instructed below.
- In Terminal 1, type: `uv run a2a_policy_agent.py`
- In Terminal 2, type: `uv run a2a_research_agent.py`

<div style="background-color:#e8f0fe; padding:15px; border-left:5px solid #4285f4; border-radius:4px">
    <b>Terminal Access:</b> Please open two new terminal windows in your Jupyter environment to run the servers.
    <br>You can typically do this by selecting <i>File -> New -> Terminal</i> from the menu.
</div>

## 5.2. Define the Sequential Workflow

Here you will:
1.  Define two `RemoteA2aAgent` instances. These act as proxies that know how to communicate with your running servers via A2A.
2.  Create a `SequentialAgent` named `root_agent`. This agent contains the logic to route the workflow through the sub-agents.
3.  Use an `InMemoryRunner` to execute the flow with a specific prompt.


In [1]:
import os

from helpers import setup_env

setup_env()

from IPython.display import Markdown, display
from google.adk.agents import SequentialAgent
from google.adk.agents.remote_a2a_agent import (
    RemoteA2aAgent,
)
from google.adk.runners import InMemoryRunner


In [2]:
host = os.getenv("AGENT_HOST")
policy_port = os.getenv("POLICY_AGENT_PORT")
research_port = os.getenv("RESEARCH_AGENT_PORT")

In [3]:
policy_agent = RemoteA2aAgent(
    name="policy_agent",
    agent_card=f"http://{host}:{policy_port}",
)
print("\tℹ️", f"{policy_agent.name} initialized")

	ℹ️ policy_agent initialized


In [4]:
health_research_agent = RemoteA2aAgent(
    name="health_research_agent",
    agent_card=f"http://{host}:{research_port}",
)
print("\tℹ️", f"{health_research_agent.name} initialized")

	ℹ️ health_research_agent initialized


In [5]:
root_agent = SequentialAgent(
    name="root_agent",
    description="Healthcare Routing Agent",
    sub_agents=[
        health_research_agent,
        policy_agent,
    ],
)
print("\tℹ️", f"{root_agent.name} initialized")


	ℹ️ root_agent initialized


## 5.3. Run the Sequential Chain

The `InMemoryRunner` executes the agent. The prompt will trigger the Research Agent to find general info, and then (sequentially) the Policy Agent to check coverage details.


In [6]:
prompt = "How can I get mental health therapy?"

In [7]:
import google.auth

try:
    credentials, project_id = google.auth.default()
    if credentials:
        print(f"✅ Credentials found!")
        print(f"Project ID: {project_id}")
        print(f"Service Account/User: {credentials.service_account_email if hasattr(credentials, 'service_account_email') else 'User Account'}")
    else:
        print("❌ No credentials found.")
except Exception as e:
    print(f"❌ Error: {e}")
    

✅ Credentials found!
Project ID: seismic-bucksaw-396319
Service Account/User: User Account


In [8]:
os.environ["GOOGLE_CLOUD_PROJECT"] = "seismic-bucksaw-396319"
# Ensure the location matches where your models are enabled
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"


In [9]:
import os

# 1. Forcefully clear any potential API Key variables from the current process
vars_to_clear = ['GEMINI_API_KEY', 'GOOGLE_API_KEY', 'PALM_API_KEY']
for var in vars_to_clear:
    if var in os.environ:
        del os.environ[var]
        print(f"🗑️ Cleared {var} from environment.")

# 2. Explicitly tell the Google SDK which project to use
# (Use the Project ID you confirmed earlier)
os.environ["GOOGLE_CLOUD_PROJECT"] = "seismic-bucksaw-396319"

# 3. Optional: Set the region if you haven't (default is usually us-central1)
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

print("✅ Environment sanitized. Only Application Default Credentials (ADC) will be used.")


🗑️ Cleared GEMINI_API_KEY from environment.
🗑️ Cleared GOOGLE_API_KEY from environment.
✅ Environment sanitized. Only Application Default Credentials (ADC) will be used.


In [10]:
import vertexai
from vertexai.generative_models import GenerativeModel

# Initialize Vertex AI
vertexai.init(project="seismic-bucksaw-396319", location="us-central1")

try:
    # Attempt a simple call to verify connectivity
    model = GenerativeModel("gemini-2.0-flash-lite-001")
    response = model.generate_content("Hello, are you active?")
    print(f"✅ Success! Response: {response.text}")
except Exception as e:
    print(f"❌ Still failing: {e}")
    

✅ Success! Response: Yes, I am active and ready to help! How can I assist you today?



In [11]:
print("Running Healthcare Workflow Agent")

# 1. Initialize the runner
runner = InMemoryRunner(root_agent)

# 2. Correctly await the debug run
# Note: In ADK 1.18+, run_debug returns a response object, not a generator
response_events = await runner.run_debug(prompt, quiet=True)

# 3. Iterate through the events in the response
for event in response_events:
    if event.is_final_response() and event.content:
        # Standard Gemini parts check
        if event.content.parts:
            display(Markdown(event.content.parts[0].text))
            

App name mismatch detected. The runner is configured with app name "InMemoryRunner", but the root agent was loaded from "/Users/ytchen/Documents/experimental/A2AWalkthrough/.venv/lib/python3.12/site-packages/google/adk/agents", which implies app name "agents".


Running Healthcare Workflow Agent


Model google/gemini-2.0-flash-lite-001 not found.

To get mental health therapy under this plan, you can access **Outpatient Services**, which include office visits. Here is how the coverage works and how to find a provider:

*   **Finding a Provider:** You can find a list of in-network providers by visiting **[includedhealth.com/google](https://includedhealth.com/google)** or by calling **(855) 431-5540**. Using an in-network provider will result in lower out-of-pocket costs.
*   **Referrals:** You do **not** need a referral to see a specialist for mental health services.
*   **Cost Sharing:** 
    *   **In-Network:** You will pay **10% coinsurance** after you have met your deductible.
    *   **Out-of-Network:** You will pay **30% coinsurance** after you have met your deductible.
*   **Deductible:** You must meet the overall plan deductible (\$1,700 for an individual or \$3,400 for a family for in-network services) before the plan begins to pay for therapy sessions.

The plan covers both outpatient office visits and inpatient services for mental health and behavioral health.

## 5.4. Resources

- [Google ADK Sequential Agents](https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/)